# 1.1.4 — Реальные объекты разжижения (альтернативный загрузчик, бывший dhfbv)

Самостоятельный загрузчик реальных объектов (резервный к `1_1_3`): ведомости читаются по
позициям колонок из `digitrock`, пиклы `CyclicModel` распаковываются безопасным unpickler
без GUI-кода, строки ведомости объединяются с массивами опытов по `lab_id`.

**Гладкая линия PPR.** Сырое поровое давление колеблется внутри цикла (квазисинусоида);
здесь берётся его **верхняя огибающая** и монотонно сглаживается общим адаптером проекта
`liquefaction_ai.data.smooth_ppr_trajectory` — в обучение идёт гладкая линия PPR(N).

На выходе — raw-артефакт и совместимый с `liquefaction_ai` population-artifact в каталоге
`data/real_objects_dhfbv`. Это **третий** источник данных единого селектора
(`synthetic` / `real_objects` / `real_objects_dhfbv`, ноутбук `1_0_select_dataset`).

In [ ]:
from __future__ import annotations

import json
import math
import pickle
import re
import sys
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

import os

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start, *start.parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    return Path("/Users/nikita/Desktop/projects/liquefaction-ai").resolve()

PROJECT_ROOT = _find_repo_root(Path.cwd())

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Корень с объектами (env LIQ_REAL_ROOT переопределяет — для проверки/переноса)
_BASE = Path(os.environ.get("LIQ_REAL_ROOT", "").strip() or
             "/Users/nikita/Desktop/Новая папка 4/Облако разжижения")
SOURCE_ROOTS = {
    "liquefaction_potential": _BASE / "Потенциал разжижения",
    "storm_liquefaction": _BASE / "Штормовое разжижение",
}
# Ограничение объектов на источник (0 = все); env LIQ_REAL_MAXOBJ для быстрой проверки
MAX_OBJECTS = int(os.environ.get("LIQ_REAL_MAXOBJ", "0")) or None

SOURCE_TO_TEST_TYPE = {
    "liquefaction_potential": "Потенциал разжижения",
    "storm_liquefaction": "Штормовое разжижение",
}

SOURCE_TO_LOAD_MODE = {
    "liquefaction_potential": "stationary_cyclic",
    "storm_liquefaction": "storm",
}

SEQ_LEN = 72
PPR_FAILURE_THRESHOLD = 0.95
STRAIN_FAILURE_THRESHOLD = 0.05
OUTPUT_DIR = PROJECT_ROOT / "data" / "real_objects_dhfbv"

# Общий адаптер гладкой линии PPR (верхняя огибающая квазисинусоиды)
from liquefaction_ai.data import smooth_ppr_trajectory

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")

## 1. Общие утилиты

Ниже небольшие функции нормализации значений, безопасного преобразования типов и выбора первого непустого значения. Они используются и для Excel, и для pickle.

In [ ]:
def is_missing(value: Any) -> bool:
    """True для None/NaN/пустых строк; безопасно для numpy-массивов и объектов."""
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == "" or value.strip().lower() in {"nan", "none"}
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def clean_scalar(value: Any) -> Any:
    """Преобразовать значение из pandas/numpy в обычный Python scalar или None."""
    if is_missing(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, str):
        value = value.replace("\xa0", " ").strip()
        return value if value else None
    return value


def safe_float(value: Any, default: float = np.nan) -> float:
    """Мягко привести число, в том числе строки с запятой, к float."""
    value = clean_scalar(value)
    if value is None:
        return float(default)
    if isinstance(value, str):
        value = value.replace(",", ".").strip()
        if not value:
            return float(default)
    try:
        return float(value)
    except Exception:
        return float(default)


def safe_int(value: Any, default: int = -1) -> int:
    """Мягко привести значение к int."""
    value = safe_float(value, np.nan)
    if not np.isfinite(value):
        return int(default)
    return int(round(value))


def normalize_lab_id(value: Any) -> Optional[str]:
    """Нормализовать лабораторный номер так, чтобы Excel и pickle совпадали по ключу."""
    value = clean_scalar(value)
    if value is None:
        return None
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return None
        return str(int(value)) if float(value).is_integer() else str(value).rstrip("0").rstrip(".")
    text = str(value).replace("\xa0", " ").strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text or None


def first_not_missing(*values: Any, default: Any = None) -> Any:
    """Вернуть первое непустое значение из списка кандидатов."""
    for value in values:
        if not is_missing(value):
            return value
    return default


def unwrap_digitrock_value(value: Any) -> Any:
    """Достать реальное значение из digitrock TestResult-like объекта с полем _value."""
    if value is None:
        return None
    if hasattr(value, "_value"):
        return clean_scalar(getattr(value, "_value"))
    return clean_scalar(value)


def finite_or_none(value: Any) -> Optional[float]:
    """Вернуть finite float или None."""
    number = safe_float(value, np.nan)
    return float(number) if np.isfinite(number) else None

## 2. Чтение ведомостей Excel

Позиции колонок взяты из `digitrock/excel_statment/position_configs.py`. Здесь они продублированы намеренно, чтобы ноутбук не зависел от импортов `digitrock`.

In [ ]:
# Основные физические свойства из digitrock PhysicalPropertyPosition.
PHYSICAL_POS = {
    "lab_id": 0,
    "borehole": 1,
    "depth": 2,
    "soil_name": 3,
    "granulometric_10": 4,
    "granulometric_5": 5,
    "granulometric_2": 6,
    "granulometric_1": 7,
    "granulometric_05": 8,
    "granulometric_025": 9,
    "granulometric_01": 10,
    "granulometric_005": 11,
    "granulometric_001": 12,
    "granulometric_0002": 13,
    "granulometric_0000": 14,
    "rs": 15,
    "r": 16,
    "rd": 17,
    "n": 18,
    "e": 19,
    "W": 20,
    "Sr": 21,
    "Wl": 22,
    "Wp": 23,
    "Ip": 24,
    "Il": 25,
    "Kf_max": 26,
    "Kf_min": 27,
    "rd_min": 28,
    "rd_max": 29,
    "Ir": 30,
    "stratigraphic_index": 33,
    "ground_water_depth": 35,
    "description": 43,
    "calcite": 143,
    "dolomite": 144,
    "Rc_p": 147,
    "ige": 148,
    "skempton_initial_statement": 229,
    "new_lab_id": 240,
}

# Механика и параметры динамики, которые нужны для обучения/контроля качества.
MECHANICAL_POS = {
    "building_pressure": 36,
    "pit_depth": 37,
    "intensity": 38,
    "frequency_vibration_creep": 39,
    "sigma_d_vibration_creep": 40,
    "acceleration": 41,
    "magnitude": 42,
    "c_cyclic_statement": 76,
    "phi_cyclic_statement": 77,
    "E_cyclic_statement": 78,
    "Cv": 80,
    "Eoed": 82,
    "Ca": 83,
    "u_0": 86,
    "p_max": 99,
    "sample_size_statement": 120,
    "compressible_thickness": 122,
    "compressible_thickness_pressure_reduce_ratio": 123,
    "sigma_1_statement": 176,
    "Pref": 177,
    "K0_ige": 178,
    "OCR": 183,
    "Eur": 190,
    "K0_oc": 206,
    "K0_nc": 207,
    "Nuur": 208,
    "c_g0_statement": 220,
    "phi_g0_statement": 221,
    "E_g0_statement": 222,
    "cycles_count_storm": 225,
    "wave_height_hw": 226,
    "frequency_storm": 227,
    "water_density_rw": 228,
    "max_deformation_statement": 230,
}

GRAIN_FIELDS = [
    "granulometric_10", "granulometric_5", "granulometric_2", "granulometric_1",
    "granulometric_05", "granulometric_025", "granulometric_01", "granulometric_005",
    "granulometric_001", "granulometric_0002", "granulometric_0000",
]

TEXT_FIELDS = {"lab_id", "borehole", "soil_name", "stratigraphic_index", "description", "ige", "new_lab_id"}


def read_statement_excel(path: Path, skiprows: int = 2) -> pd.DataFrame:
    """Прочитать ведомость .xls/.xlsx и вернуть одну строку на лабораторный образец."""
    path = Path(path)
    engine = "xlrd" if path.suffix.lower() == ".xls" else None
    raw = pd.read_excel(path, engine=engine, usecols="A:IV", skiprows=skiprows)
    records: List[Dict[str, Any]] = []
    positions = {**PHYSICAL_POS, **MECHANICAL_POS}

    for row_index, row in raw.iterrows():
        lab_id = normalize_lab_id(row.iloc[PHYSICAL_POS["lab_id"]])
        if lab_id is None or lab_id == "0":
            continue

        record: Dict[str, Any] = {
            "lab_id": lab_id,
            "statement_row_index": int(row_index),
            "statement_file": str(path),
        }
        for name, position in positions.items():
            if position >= len(row):
                record[name] = None
                continue
            value = clean_scalar(row.iloc[position])
            if name == "lab_id":
                record[name] = lab_id
            elif name in TEXT_FIELDS:
                record[name] = normalize_lab_id(value) if name in {"borehole", "new_lab_id"} else value
            else:
                record[name] = safe_float(value, np.nan)
        records.append(record)

    return pd.DataFrame(records)


def estimate_grain_percentile(row: pd.Series, target_percent: float) -> float:
    """Оценить Dp по грансоставу; нужна только для Cu как запасной вариант."""
    bounds = {
        "granulometric_10": 20.0,
        "granulometric_5": 10.0,
        "granulometric_2": 5.0,
        "granulometric_1": 2.0,
        "granulometric_05": 1.0,
        "granulometric_025": 0.5,
        "granulometric_01": 0.25,
        "granulometric_005": 0.1,
        "granulometric_001": 0.05,
        "granulometric_0002": 0.01,
        "granulometric_0000": 0.002,
    }
    fractions = np.array([safe_float(row.get(k), 0.0) for k in bounds], dtype=float)
    fractions = np.nan_to_num(fractions, nan=0.0)
    total = fractions.sum()
    if total <= 1e-9:
        return np.nan
    fractions = fractions / total * 100.0
    diameters = np.array(list(bounds.values()), dtype=float)
    passing = np.cumsum(fractions[::-1])[::-1]
    d_asc = diameters[::-1]
    p_asc = np.maximum.accumulate(np.clip(passing[::-1], 0.0, 100.0))
    target = float(np.clip(target_percent, p_asc.min(), p_asc.max()))
    if np.unique(p_asc).size < 2:
        return float(d_asc[0])
    return float(np.exp(np.interp(target, p_asc, np.log(d_asc))))


def estimate_cu(row: pd.Series, default: float = 5.0) -> float:
    """Оценить коэффициент неоднородности Cu = D60 / D10 по грансоставу."""
    d10 = estimate_grain_percentile(row, 10.0)
    d60 = estimate_grain_percentile(row, 60.0)
    if not np.isfinite(d10) or not np.isfinite(d60) or d10 <= 0:
        return default
    return float(np.clip(d60 / d10, 1.0, 200.0))


def infer_type_ground_from_name(name: Any) -> int:
    """Грубый fallback для type_ground 1..9 по русскому названию грунта."""
    text = str(name or "").lower()
    if "торф" in text:
        return 9
    if "суглин" in text:
        return 7
    if "супес" in text:
        return 6
    if "глин" in text:
        return 8
    if "пылеват" in text:
        return 5
    if "мелк" in text:
        return 4
    if "сред" in text:
        return 3
    if "круп" in text:
        return 2
    if "гравел" in text or "пес" in text:
        return 1
    return 5

## 3. Безопасное чтение `CyclicModel` pickle

`data CyclicModel - 1.pickle` и `handler CyclicModel - 1.pickle` содержат экземпляры классов `digitrock`. Для подготовки датасета нам не нужны методы этих классов, нужны только их поля. Поэтому unpickler ниже заменяет неизвестные классы простыми контейнерами и не импортирует `digitrock`.

In [ ]:
class GenericDigitrockObject:
    """Контейнер для объектов digitrock, восстановленных из pickle без импорта класса."""

    def __setstate__(self, state: Any) -> None:
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["state"] = state

    def __repr__(self) -> str:
        keys = ", ".join(sorted(self.__dict__.keys())[:8])
        suffix = "..." if len(self.__dict__) > 8 else ""
        return f"{type(self).__name__}({keys}{suffix})"


_PICKLE_CLASS_CACHE: Dict[Tuple[str, str], type] = {}


def make_placeholder_class(module: str, name: str) -> type:
    """Создать стабильный placeholder-класс для конкретного module.name из pickle."""
    key = (module, name)
    if key not in _PICKLE_CLASS_CACHE:
        _PICKLE_CLASS_CACHE[key] = type(name, (GenericDigitrockObject,), {"__module__": module})
    return _PICKLE_CLASS_CACHE[key]


class SafeDigitrockUnpickler(pickle.Unpickler):
    """Unpickler, разрешающий numpy/builtins и заменяющий остальные классы контейнерами."""

    SAFE_PREFIXES = ("numpy", "builtins", "collections", "datetime")

    def find_class(self, module: str, name: str) -> Any:
        if module.startswith(self.SAFE_PREFIXES):
            return super().find_class(module, name)
        return make_placeholder_class(module, name)


def load_digitrock_pickle(path: Path) -> Dict[str, Any]:
    """Загрузить pickle digitrock без импортов digitrock."""
    with Path(path).open("rb") as file:
        payload = SafeDigitrockUnpickler(file).load()
    if not isinstance(payload, dict) or "data" not in payload:
        raise ValueError(f"Неожиданный формат pickle: {path}")
    return payload


def object_to_flat_dict(obj: Any, prefix: str, fields: Iterable[str]) -> Dict[str, Any]:
    """Достать перечисленные поля объекта в плоский dict с prefix."""
    result = {}
    for field in fields:
        result[f"{prefix}_{field}"] = clean_scalar(getattr(obj, field, None)) if obj is not None else None
    return result


HANDLER_PARAM_FIELDS = [
    "frequency", "points_in_cycle", "cycles_count", "n_fail", "sigma_1", "sigma_3", "t", "K0",
    "c", "fi", "E", "E0", "qf", "qf_2", "Cv", "damping_ratio", "ground_type", "Kd",
    "reconsolidation_time", "len_cycles", "reverse",
]

HANDLER_PHYSICAL_FIELDS = [
    "laboratory_number", "borehole", "depth", "soil_name", "ige", "rs", "r", "rd", "n", "e", "W",
    "Sr", "Wl", "Wp", "Ip", "Il", "Ir", "stratigraphic_index", "ground_water_depth",
    "granulometric_10", "granulometric_5", "granulometric_2", "granulometric_1", "granulometric_05",
    "granulometric_025", "granulometric_01", "granulometric_005", "granulometric_001",
    "granulometric_0002", "granulometric_0000", "type_ground", "sample_size", "skempton_initial",
    "description", "D50", "Cu", "plaxis_gran_classification",
]

HANDLER_RESULT_FIELDS = [
    "max_PPR", "max_strain", "max_shear_strain", "fail_cycle", "fail_cycle_criterion_PPR",
    "fail_cycle_criterion_strain", "fail_cycle_criterion_stress", "conclusion", "damping_ratio",
    "relative_scattered_energy", "dynamic_stability", "G", "transverse_waves_velocity", "E0",
    "t_rel_static", "t_rel_dynamic", "K_ds", "u_excess",
]


def read_handler_pickle(path: Path) -> pd.DataFrame:
    """Прочитать handler-pickle и вернуть табличные параметры/результаты по lab_id."""
    payload = load_digitrock_pickle(path)
    rows: List[Dict[str, Any]] = []
    for raw_lab_id, handler in payload["data"].items():
        lab_id = normalize_lab_id(raw_lab_id)
        params = getattr(handler, "_test_params", None)
        physical = getattr(params, "physical", None)
        result_obj = getattr(handler, "_test_result", None)
        draw_params = getattr(handler, "_draw_params", None)

        row: Dict[str, Any] = {
            "lab_id": lab_id,
            "handler_file": str(path),
            "handler_version": payload.get("version"),
        }
        row.update(object_to_flat_dict(params, "handler", HANDLER_PARAM_FIELDS))
        row.update(object_to_flat_dict(physical, "handler_physical", HANDLER_PHYSICAL_FIELDS))
        row.update(object_to_flat_dict(draw_params, "draw", ["PPR_max", "strain_max", "cycles_count", "PPR_slant", "strain_slant"]))

        for field in HANDLER_RESULT_FIELDS:
            row[f"result_{field}"] = unwrap_digitrock_value(getattr(result_obj, field, None)) if result_obj is not None else None
        rows.append(row)
    return pd.DataFrame(rows)


def read_data_pickle(path: Path) -> Dict[str, Any]:
    """Прочитать data-pickle и вернуть словарь lab_id -> объект с массивами опыта."""
    payload = load_digitrock_pickle(path)
    return {normalize_lab_id(key): value for key, value in payload["data"].items()}

## 4. Поиск объектов и сбор строк датасета

In [ ]:
def discover_objects(source_roots: Dict[str, Path]) -> pd.DataFrame:
    """Найти объектные папки, ведомость Excel и pickle-файлы внутри каждого объекта."""
    rows: List[Dict[str, Any]] = []
    for source_key, root in source_roots.items():
        root = Path(root)
        if not root.exists():
            raise FileNotFoundError(root)
        for object_dir in sorted(path for path in root.iterdir() if path.is_dir()):
            excel_files = sorted(
                p for p in list(object_dir.glob("*.xls")) + list(object_dir.glob("*.xlsx"))
                if not p.name.startswith("~$")
            )
            data_pickles = sorted(object_dir.glob("*/data CyclicModel*.pickle"))
            handler_pickles = sorted(object_dir.glob("*/handler CyclicModel*.pickle"))
            rows.append({
                "source_key": source_key,
                "test_type": SOURCE_TO_TEST_TYPE[source_key],
                "load_mode": SOURCE_TO_LOAD_MODE[source_key],
                "object_name": object_dir.name,
                "object_dir": str(object_dir),
                "statement_path": str(excel_files[0]) if excel_files else None,
                "data_pickle_path": str(data_pickles[0]) if data_pickles else None,
                "handler_pickle_path": str(handler_pickles[0]) if handler_pickles else None,
                "excel_count": len(excel_files),
                "data_pickle_count": len(data_pickles),
                "handler_pickle_count": len(handler_pickles),
            })
    objects = pd.DataFrame(rows)
    missing = objects[objects[["statement_path", "data_pickle_path"]].isna().any(axis=1)]
    if len(missing):
        print("Объекты без обязательных файлов:")
        display(missing[["source_key", "object_name", "statement_path", "data_pickle_path", "handler_pickle_path"]])
    return objects


objects_df = discover_objects(SOURCE_ROOTS)
print(f"Найдено объектов: {len(objects_df)}")
display(objects_df[["source_key", "object_name", "excel_count", "data_pickle_count", "handler_pickle_count"]])

In [ ]:
def make_target_cycles(raw_cycles: np.ndarray, seq_len: int) -> np.ndarray:
    """Построить регулярную сетку циклов от первого до последнего измерения."""
    raw_cycles = np.asarray(raw_cycles, dtype=float)
    raw_cycles = raw_cycles[np.isfinite(raw_cycles)]
    if raw_cycles.size == 0:
        return np.linspace(0.0, 1.0, seq_len, dtype=np.float32)
    start = float(np.nanmin(raw_cycles))
    stop = float(np.nanmax(raw_cycles))
    if not np.isfinite(stop) or stop <= start:
        stop = start + 1.0
    return np.linspace(start, stop, seq_len, dtype=np.float32)


def prepare_xy_for_interp(cycles: np.ndarray, values: Any) -> Tuple[np.ndarray, np.ndarray]:
    """Очистить x/y, удалить NaN, отсортировать и убрать повторяющиеся x."""
    x = np.asarray(cycles, dtype=float)
    y = np.asarray(values, dtype=float) if values is not None else np.zeros_like(x, dtype=float)
    size = min(x.size, y.size)
    x, y = x[:size], y[:size]
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if x.size == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 0.0])
    order = np.argsort(x)
    x, y = x[order], y[order]
    unique_x, unique_idx = np.unique(x, return_index=True)
    return unique_x, y[unique_idx]


def resample_signal(cycles: np.ndarray, values: Any, target_cycles: np.ndarray, fill: float = 0.0) -> np.ndarray:
    """Ресэмплировать сигнал на target_cycles линейной интерполяцией."""
    x, y = prepare_xy_for_interp(cycles, values)
    if x.size == 1:
        return np.full_like(target_cycles, y[0], dtype=np.float32)
    result = np.interp(target_cycles, x, y, left=y[0], right=y[-1]).astype(np.float32)
    result[~np.isfinite(result)] = fill
    return result


def get_data_array(data_obj: Any, name: str) -> np.ndarray:
    """Достать массив из объекта CyclicData; если поля нет, вернуть пустой массив."""
    value = getattr(data_obj, name, None)
    if value is None:
        return np.array([], dtype=np.float32)
    return np.asarray(value, dtype=np.float32)


def infer_csr_base(row: pd.Series, raw_deviator: np.ndarray) -> float:
    """Определить CSR_base: сперва из handler t/sigma_1, затем из амплитуды девиатора."""
    sigma_1 = safe_float(first_not_missing(row.get("handler_sigma_1"), row.get("sigma_1_statement")), np.nan)
    t = safe_float(row.get("handler_t"), np.nan)
    if np.isfinite(sigma_1) and abs(sigma_1) > 1e-9 and np.isfinite(t):
        return float(np.clip(abs(t / sigma_1), 0.0, 2.0))
    if raw_deviator.size and np.isfinite(sigma_1) and abs(sigma_1) > 1e-9:
        amp = (np.nanmax(raw_deviator) - np.nanmin(raw_deviator)) / 2.0
        return float(np.clip(abs(amp / sigma_1), 0.0, 2.0))
    return 0.0


def infer_liq_label(row: pd.Series, ppr: np.ndarray, strain: np.ndarray) -> int:
    """Определить бинарную метку разжижения по заключению, fail_cycle и порогам PPR/strain."""
    conclusion = str(row.get("result_conclusion") or "").lower()
    if "не разжиж" in conclusion or "неразжиж" in conclusion:
        return 0
    if "разжиж" in conclusion:
        return 1
    fail_cycle = finite_or_none(row.get("result_fail_cycle"))
    if fail_cycle is not None and fail_cycle > 0:
        return 1
    ppr_max = np.nanmax(ppr) if ppr.size else np.nan
    strain_abs_max = np.nanmax(np.abs(strain)) if strain.size else np.nan
    return int((np.isfinite(ppr_max) and ppr_max >= PPR_FAILURE_THRESHOLD) or
               (np.isfinite(strain_abs_max) and strain_abs_max >= STRAIN_FAILURE_THRESHOLD))


def infer_n_liq(row: pd.Series, cycles: np.ndarray, ppr: np.ndarray, strain: np.ndarray) -> float:
    """Определить N_liq: handler fail_cycle или первый переход PPR/strain через критерий."""
    for field in ["result_fail_cycle", "result_fail_cycle_criterion_PPR", "result_fail_cycle_criterion_strain", "handler_n_fail"]:
        value = finite_or_none(row.get(field))
        if value is not None and value > 0:
            return float(value)
    mask = (ppr >= PPR_FAILURE_THRESHOLD) | (np.abs(strain) >= STRAIN_FAILURE_THRESHOLD)
    if mask.any():
        return float(cycles[int(np.argmax(mask))])
    return float(cycles[-1]) if cycles.size else 1.0


def build_object_dataset(object_row: pd.Series, seq_len: int = SEQ_LEN) -> Tuple[pd.DataFrame, Dict[str, List[np.ndarray]]]:
    """Собрать все опыты одного объекта в metadata-таблицу и списки массивов."""
    statement = read_statement_excel(Path(object_row["statement_path"]))
    handler = read_handler_pickle(Path(object_row["handler_pickle_path"])) if object_row.get("handler_pickle_path") else pd.DataFrame()
    data_map = read_data_pickle(Path(object_row["data_pickle_path"]))

    merged = statement.merge(handler, on="lab_id", how="left") if len(handler) else statement.copy()
    merged = merged[merged["lab_id"].isin(data_map.keys())].copy()

    meta_rows: List[Dict[str, Any]] = []
    arrays: Dict[str, List[np.ndarray]] = {key: [] for key in [
        "cycles", "csr", "ppr", "strain", "deviator", "mean_effective_stress", "cell_pressure", "valid_mask"
    ]}

    for _, row in merged.iterrows():
        lab_id = row["lab_id"]
        data_obj = data_map[lab_id]
        raw_cycles = get_data_array(data_obj, "cycles")
        raw_ppr = get_data_array(data_obj, "PPR")
        if raw_cycles.size == 0 or raw_ppr.size == 0:
            continue

        raw_strain = get_data_array(data_obj, "strain")
        raw_deviator = get_data_array(data_obj, "deviator")
        raw_mean_eff = get_data_array(data_obj, "mean_effective_stress")
        raw_cell = get_data_array(data_obj, "cell_pressure")

        # Гладкая линия PPR(N): верхняя огибающая квазисинусоиды + монотонное сглаживание
        target_cycles, ppr, valid_mask = smooth_ppr_trajectory(raw_cycles, raw_ppr, seq_len)
        ppr = np.clip(ppr, 0.0, 1.05)
        strain = resample_signal(raw_cycles, raw_strain, target_cycles)
        deviator = resample_signal(raw_cycles, raw_deviator, target_cycles)
        mean_eff = resample_signal(raw_cycles, raw_mean_eff, target_cycles)
        cell = resample_signal(raw_cycles, raw_cell, target_cycles)
        csr_base = infer_csr_base(row, raw_deviator)
        csr = np.full(seq_len, csr_base, dtype=np.float32)
        liq_label = infer_liq_label(row, ppr, strain)
        n_liq = infer_n_liq(row, target_cycles, ppr, strain)

        meta = row.to_dict()
        meta.update({
            "source_key": object_row["source_key"],
            "test_type": object_row["test_type"],
            "load_mode": object_row["load_mode"],
            "object_name": object_row["object_name"],
            "object_dir": object_row["object_dir"],
            "raw_points": int(raw_cycles.size),
            "raw_cycle_min": float(np.nanmin(raw_cycles)),
            "raw_cycle_max": float(np.nanmax(raw_cycles)),
            "raw_ppr_min": float(np.nanmin(raw_ppr)),
            "raw_ppr_max": float(np.nanmax(raw_ppr)),
            "CSR_base": csr_base,
            "liq_label": liq_label,
            "N_liq": n_liq,
            "seq_len": seq_len,
        })
        meta_rows.append(meta)
        arrays["cycles"].append(target_cycles.astype(np.float32))
        arrays["csr"].append(csr)
        arrays["ppr"].append(ppr.astype(np.float32))
        arrays["strain"].append(strain.astype(np.float32))
        arrays["deviator"].append(deviator.astype(np.float32))
        arrays["mean_effective_stress"].append(mean_eff.astype(np.float32))
        arrays["cell_pressure"].append(cell.astype(np.float32))
        arrays["valid_mask"].append(valid_mask)

    return pd.DataFrame(meta_rows), arrays


def build_full_dataset(objects: pd.DataFrame, seq_len: int = SEQ_LEN) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    """Собрать датасет по всем найденным объектам."""
    all_meta: List[pd.DataFrame] = []
    all_arrays: Dict[str, List[np.ndarray]] = {key: [] for key in [
        "cycles", "csr", "ppr", "strain", "deviator", "mean_effective_stress", "cell_pressure", "valid_mask"
    ]}

    usable = objects.dropna(subset=["statement_path", "data_pickle_path"])
    if MAX_OBJECTS:
        usable = usable.groupby("source_key", group_keys=False).head(MAX_OBJECTS)
    for _, object_row in usable.iterrows():
        meta, arrays = build_object_dataset(object_row, seq_len=seq_len)
        all_meta.append(meta)
        for key, values in arrays.items():
            all_arrays[key].extend(values)
        print(f"{object_row['source_key']} | {object_row['object_name']}: {len(meta)} опытов")

    records = pd.concat(all_meta, ignore_index=True) if all_meta else pd.DataFrame()
    stacked = {key: np.stack(values, axis=0).astype(np.float32) for key, values in all_arrays.items()}
    stacked["liq_label"] = records["liq_label"].to_numpy(dtype=np.float32)
    stacked["n_liq"] = records["N_liq"].to_numpy(dtype=np.float32)
    return records, stacked


records_df, arrays = build_full_dataset(objects_df, seq_len=SEQ_LEN)
print("\nИтоговая форма массивов:")
for key, value in arrays.items():
    print(f"{key:22s} {value.shape} {value.dtype}")

display(records_df.groupby(["source_key", "object_name"]).size().rename("n_tests").reset_index())

## 5. Подготовка таблиц в формат проекта `liquefaction_ai`

Эти функции приводят объединённые строки к `soil_df` и `load_df`, которые понимает `liquefaction_ai.data.real_adapter.build_population_from_experiments`.

In [ ]:
def as_numeric_series(df: pd.DataFrame, *columns: str, default: float = np.nan) -> pd.Series:
    """Взять первую существующую числовую колонку из списка, затем заполнить default."""
    result = pd.Series(np.nan, index=df.index, dtype=float)
    for column in columns:
        if column in df.columns:
            candidate = pd.to_numeric(df[column], errors="coerce")
            result = result.where(result.notna(), candidate)
    if not np.isnan(default):
        result = result.fillna(default)
    return result.astype(float)


def normalize_fraction_percent(series: pd.Series) -> pd.Series:
    """Оставить процентную долю в диапазоне 0..100."""
    return pd.to_numeric(series, errors="coerce").fillna(0.0).clip(lower=0.0, upper=100.0)


def derive_relative_density(df: pd.DataFrame) -> pd.Series:
    """D_r в долях единицы: Ir, если есть, иначе оценка по e."""
    ir = as_numeric_series(df, "handler_physical_Ir", "Ir", default=np.nan)
    dr = ir.copy()
    dr = dr.where(dr <= 1.5, dr / 100.0)
    e = as_numeric_series(df, "e", "handler_physical_e", default=np.nan)
    e_based = ((1.05 - e) / (1.05 - 0.42)).clip(0.05, 0.95)
    dr = dr.where(dr.notna(), e_based)
    return dr.fillna(0.5).clip(0.02, 0.98)


def derive_vs(df: pd.DataFrame) -> pd.Series:
    """Скорость V_s в масштабе проекта; малые digitrock-значения масштабируются до м/с-like диапазона."""
    vs = as_numeric_series(df, "result_transverse_waves_velocity", default=np.nan)
    vs = vs.where(vs >= 10.0, vs * 100.0)
    e0 = as_numeric_series(df, "result_E0", "handler_E0", default=np.nan)
    density = as_numeric_series(df, "r", "handler_physical_r", default=1.8) * 1000.0
    # fallback: E0 в МПа/кПа встречается в разных местах, поэтому используем только как грубую оценку.
    e0_mpa = e0.where(e0 < 1000.0, e0 / 1000.0)
    vs_from_e0 = np.sqrt(np.maximum(e0_mpa * 1_000_000.0 / np.maximum(2.6 * density, 1.0), 1.0))
    vs = vs.where(vs.notna(), vs_from_e0)
    return vs.fillna(180.0).clip(40.0, 500.0)


def derive_permeability(df: pd.DataFrame) -> pd.Series:
    """Оценить permeability в м/с из Kf_min/Kf_max или задать консервативный default."""
    kf_max = as_numeric_series(df, "Kf_max", default=np.nan)
    kf_min = as_numeric_series(df, "Kf_min", default=np.nan)
    kf = pd.concat([kf_min, kf_max], axis=1).mean(axis=1, skipna=True)
    # В ведомостях Kf часто в м/сут или отсутствует; положительные большие значения переводим в м/с приближенно.
    kf = kf.where(kf <= 1.0, kf / 86400.0)
    return kf.where(kf > 0.0, np.nan).fillna(1e-5).clip(1e-9, 1e-2)


def build_project_soil_df(records: pd.DataFrame) -> pd.DataFrame:
    """Собрать soil_df с обязательными колонками проекта."""
    from liquefaction_ai.constants import SOIL_NAMES

    df = records.copy()
    type_ground = as_numeric_series(df, "handler_physical_type_ground", "handler_ground_type", default=np.nan)
    inferred = df.get("soil_name", pd.Series(index=df.index, dtype=object)).map(infer_type_ground_from_name)
    type_ground = type_ground.where(type_ground.notna(), inferred).fillna(5).round().astype(int).clip(1, 9)

    fines_content = sum(normalize_fraction_percent(df.get(field, 0.0)) for field in [
        "granulometric_01", "granulometric_005", "granulometric_001", "granulometric_0002", "granulometric_0000"
    ]).clip(0.0, 100.0)
    clay_fraction = sum(normalize_fraction_percent(df.get(field, 0.0)) for field in [
        "granulometric_0002", "granulometric_0000"
    ]).clip(0.0, 100.0)

    cu_values = []
    for _, row in df.iterrows():
        cu_values.append(first_not_missing(row.get("handler_physical_Cu"), estimate_cu(row), default=5.0))

    soil = pd.DataFrame(index=df.index)
    soil["lab_id"] = df["lab_id"].astype(str)
    soil["object_name"] = df["object_name"]
    soil["test_type"] = df["test_type"]
    soil["soil_type"] = [SOIL_NAMES[i - 1] for i in type_ground]
    soil["type_ground"] = type_ground.astype(int)
    soil["class_id"] = (type_ground - 1).astype(int)
    soil["e"] = as_numeric_series(df, "e", "handler_physical_e", default=0.75).clip(0.2, 2.0)
    soil["D_r"] = derive_relative_density(df)
    soil["I_p"] = as_numeric_series(df, "Ip", "handler_physical_Ip", default=0.0).fillna(0.0).clip(0.0, 80.0)
    soil["Il"] = as_numeric_series(df, "Il", "handler_physical_Il", default=0.0).fillna(0.0).clip(-1.0, 2.0)
    soil["Ir"] = as_numeric_series(df, "Ir", "handler_physical_Ir", default=0.0).fillna(0.0)
    soil["V_s"] = derive_vs(df)
    soil["Vs1"] = soil["V_s"]
    damping = as_numeric_series(df, "handler_damping_ratio", "result_damping_ratio", default=3.0)
    soil["xi"] = damping.where(damping <= 1.0, damping / 100.0).clip(0.001, 0.30)
    soil["damping_ratio"] = soil["xi"]
    soil["sigma_eff"] = as_numeric_series(df, "handler_sigma_3", default=100.0).clip(1.0, 2000.0)
    soil["permeability"] = derive_permeability(df)
    soil["fines_content"] = fines_content
    soil["clay_fraction"] = clay_fraction
    soil["Cu"] = pd.Series(cu_values, index=df.index, dtype=float).fillna(5.0).clip(1.0, 200.0)
    soil["OCR"] = as_numeric_series(df, "OCR", default=1.0).fillna(1.0).clip(0.5, 20.0)
    soil["K0"] = as_numeric_series(df, "handler_K0", "K0_nc", "K0_ige", default=0.5).fillna(0.5).clip(0.1, 2.0)
    soil["static_shear_ratio"] = 0.0
    soil["saturation"] = as_numeric_series(df, "Sr", "handler_physical_Sr", default=1.0).where(lambda s: s <= 1.5, lambda s: s / 100.0).clip(0.0, 1.0)
    soil["B_value"] = as_numeric_series(df, "skempton_initial_statement", "handler_physical_skempton_initial", default=0.95).clip(0.0, 1.0)
    soil["aging_years"] = 1.0
    soil["cementation_index"] = 0.0
    return soil.reset_index(drop=True)


def build_project_load_df(records: pd.DataFrame) -> pd.DataFrame:
    """Собрать load_df с обязательными колонками проекта."""
    from liquefaction_ai.constants import LOAD_NAMES

    df = records.copy()
    mode_names = df["load_mode"].fillna("stationary_cyclic").astype(str)
    mode_id = mode_names.map({name: idx for idx, name in enumerate(LOAD_NAMES)}).fillna(LOAD_NAMES.index("stationary_cyclic")).astype(int)
    frequency = as_numeric_series(df, "handler_frequency", "frequency_storm", "frequency_vibration_creep", default=0.5).clip(0.01, 50.0)
    n_max = as_numeric_series(df, "raw_cycle_max", "handler_cycles_count", "cycles_count_storm", default=SEQ_LEN).clip(1.0, 100_000.0)

    load = pd.DataFrame(index=df.index)
    # lab_id хранится в soil_df; в load_df его не оставляем, чтобы meta после concat не имела дублей.
    load["load_mode"] = mode_names
    load["mode_id"] = mode_id
    load["CSR_base"] = pd.to_numeric(df["CSR_base"], errors="coerce").fillna(0.0).clip(0.0, 2.0)
    load["frequency"] = frequency
    load["amp_scale"] = 1.0
    load["N_max"] = n_max
    load["nonstationarity"] = 0.0
    return load.reset_index(drop=True)


soil_df = build_project_soil_df(records_df)
load_df = build_project_load_df(records_df)

print("soil_df", soil_df.shape)
print("load_df", load_df.shape)
display(soil_df.head())
display(load_df.head())

## 6. Сохранение raw-датасета и population-artifact

In [ ]:
def make_parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Привести смешанные object-колонки к parquet-safe строкам/JSON."""
    safe = df.copy()
    for column in safe.columns:
        if safe[column].dtype != "object":
            continue

        def convert(value: Any) -> Any:
            if is_missing(value):
                return None
            if isinstance(value, (dict, list, tuple, set)):
                return json.dumps(value, ensure_ascii=False, default=str)
            return str(value)

        safe[column] = safe[column].map(convert)
    return safe


def save_raw_dataset(records: pd.DataFrame, arrays: Dict[str, np.ndarray], output_dir: Path) -> Dict[str, Path]:
    """Сохранить объединённые метаданные и массивы без потери исходных полей."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    records_path = output_dir / "records.parquet"
    arrays_path = output_dir / "arrays_raw.npz"
    schema_path = output_dir / "schema.json"
    csv_fallback_path = output_dir / "records.csv"

    parquet_records = make_parquet_safe(records)
    parquet_records.to_parquet(records_path, index=False)
    records.to_csv(csv_fallback_path, index=False)
    np.savez_compressed(arrays_path, **arrays)
    schema = {
        "description": "Raw cyclic liquefaction dataset prepared by dhfbv.ipynb",
        "n_experiments": int(len(records)),
        "seq_len": int(arrays["cycles"].shape[1]),
        "array_shapes": {key: list(value.shape) for key, value in arrays.items()},
        "source_roots": {key: str(value) for key, value in SOURCE_ROOTS.items()},
        "failure_thresholds": {
            "PPR": PPR_FAILURE_THRESHOLD,
            "abs_strain": STRAIN_FAILURE_THRESHOLD,
        },
    }
    schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"records": records_path, "records_csv": csv_fallback_path, "arrays": arrays_path, "schema": schema_path}


def save_project_population_artifact(
    records: pd.DataFrame,
    arrays: Dict[str, np.ndarray],
    output_dir: Path,
) -> Tuple[Dict[str, Any], Any, Path]:
    """Собрать и сохранить artifact, совместимый с обучающим пайплайном liquefaction_ai."""
    from liquefaction_ai.config import get_default_config
    from liquefaction_ai.data.io import save_population_artifact
    from liquefaction_ai.data.real_adapter import build_population_from_experiments

    config = get_default_config()
    config.seq_len = int(arrays["cycles"].shape[1])
    config.benchmark_subset = min(int(config.benchmark_subset), int(len(records)))
    config.ablation_subset = min(int(config.ablation_subset), int(len(records)))

    soil = build_project_soil_df(records)
    load = build_project_load_df(records)

    population = build_population_from_experiments(
        soil_df=soil,
        load_df=load,
        cycles=arrays["cycles"],
        csr=arrays["csr"],
        r_measured=arrays["ppr"],
        valid_mask=arrays["valid_mask"],
        liq_label=arrays["liq_label"],
        n_liq=arrays["n_liq"],
        config=config,
    )

    artifact_dir = Path(output_dir)  # артефакт прямо в data/real_objects_dhfbv (для селектора)
    save_population_artifact(artifact_dir, population, config)
    return population, config, artifact_dir


paths = save_raw_dataset(records_df, arrays, OUTPUT_DIR)
print("Raw dataset saved:")
for name, path in paths.items():
    print(f"  {name:12s}: {path}")

population, config, artifact_dir = save_project_population_artifact(records_df, arrays, OUTPUT_DIR)
print(f"\nPopulation artifact saved: {artifact_dir}")
print(f"n = {len(population['meta'])}, seq_len = {config.seq_len}, benchmark_subset = {config.benchmark_subset}")
print("static_features", population["static_features"].shape)
print("seq_inputs     ", population["seq_inputs"].shape)

## 7. Контроль качества

In [ ]:
def dataset_quality_report(records: pd.DataFrame, arrays: Dict[str, np.ndarray], population: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Краткий QA-отчёт по собранному датасету."""
    report = {
        "n_experiments": int(len(records)),
        "n_objects": int(records["object_name"].nunique()),
        "seq_len": int(arrays["cycles"].shape[1]),
        "liq_rate": float(records["liq_label"].mean()),
        "ppr_range": [float(np.nanmin(arrays["ppr"])), float(np.nanmax(arrays["ppr"]))],
        "cycle_range": [float(np.nanmin(arrays["cycles"])), float(np.nanmax(arrays["cycles"]))],
        "csr_range": [float(np.nanmin(arrays["csr"])), float(np.nanmax(arrays["csr"]))],
        "nan_counts_arrays": {key: int(np.isnan(value).sum()) for key, value in arrays.items() if np.issubdtype(value.dtype, np.floating)},
    }
    if population is not None:
        report["population_static_features_shape"] = list(population["static_features"].shape)
        report["population_seq_inputs_shape"] = list(population["seq_inputs"].shape)
        report["population_meta_columns"] = int(len(population["meta"].columns))
    return report


qa = dataset_quality_report(records_df, arrays, population)
print(json.dumps(qa, ensure_ascii=False, indent=2))

display(records_df.groupby(["source_key", "liq_label"]).size().rename("n").reset_index())
display(records_df[["object_name", "lab_id", "test_type", "CSR_base", "raw_cycle_max", "raw_ppr_max", "liq_label", "N_liq"]].head(12))

## 8. Быстрая визуальная проверка нескольких траекторий

In [ ]:
import matplotlib.pyplot as plt

sample_idx = np.linspace(0, len(records_df) - 1, min(8, len(records_df)), dtype=int)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for idx in sample_idx:
    label = f"{records_df.loc[idx, 'object_name']} | {records_df.loc[idx, 'lab_id']}"
    axes[0].plot(arrays["cycles"][idx], arrays["ppr"][idx], alpha=0.8, label=label)
    axes[1].plot(arrays["cycles"][idx], arrays["strain"][idx], alpha=0.8, label=label)
axes[0].axhline(PPR_FAILURE_THRESHOLD, color="crimson", linestyle="--", linewidth=1)
axes[0].set_ylabel("PPR")
axes[1].axhline(STRAIN_FAILURE_THRESHOLD, color="crimson", linestyle="--", linewidth=1)
axes[1].axhline(-STRAIN_FAILURE_THRESHOLD, color="crimson", linestyle="--", linewidth=1)
axes[1].set_ylabel("strain")
axes[1].set_xlabel("cycles")
axes[0].legend(fontsize=7, loc="best")
fig.tight_layout()
plt.show()

## 9. Как использовать дальше

Артефакт для обучения сохранён прямо в каталоге источника:

`data/real_objects_dhfbv/`

Его подхватывает единый селектор датасетов (ноутбук `1_0_select_dataset`, источник
`real_objects_dhfbv`) или можно загрузить напрямую:

```python
from liquefaction_ai.data.io import load_population_artifact
population, config = load_population_artifact("data/real_objects_dhfbv")
```

Рядом сохранён raw-слой: `records.parquet`/`records.csv` (ведомость + handler + диагностика),
`arrays_raw.npz` (ресэмплированные массивы), `schema.json`.